## ST115 PROJECT - Web Scraping

This notebook describes the steps (with corresponding code) that were taken to gather the data for the `ST_courses_20/21` and `ST_courses_17/18` files. The data was collected in three parts: 

* <i>Part 1</i>: data for the 2020/21 academic year with every step explained in great detail. 
* <i>Part 2</i>: data for the 2017/18 academic year with fewer comments two avoid repetition. 
* <i>Part 3</i>: save the data from <i>Parts 1 and 2</i> in `csv` format. 

## Part 1: 2020/21 academic year

**Step 1: Import required libraries**

In [1]:
import requests #use to send HTTP requests and obtain the response
from bs4 import BeautifulSoup #use to extract information from html
import pandas as pd

**Step 2: Obtain a list of all courses from Source 2**

In [2]:
url_courses_list = 'https://www.lse.ac.uk/resources/calendar2020-2021/courseGuides/undergraduate.htm#generated-subheading19'
r_courses_list = requests.get(url_courses_list)
soup_courses_list = BeautifulSoup(r_courses_list.content,'lxml')
list_of_courses = pd.read_html(r_courses_list.content) #list of all LSE courses

**Step 3: Create a dataframe with course code and name**

In [3]:
st_courses = list_of_courses[18] #extract ST courses
st_courses['Course_code'] = st_courses['ST'].str.split().str[:1].str.join('') #add a column with course code info
st_courses['Course_name'] = st_courses['ST'].str.split().str[1:].str.join(' ') #add a column with course name info
st_courses.drop('ST', axis = 1, inplace = True) #remove the 'ST' column to avoid repeating info
st_courses.set_index('Course_code', inplace = True)

**Step 4: Create a temporary dataframe with grade distribution and merge it with the dataframe from Step 3**

Get a list of all ST courses and create a temporary dataframe:

In [4]:
st_courses_list = st_courses.index.tolist() 
st_courses_marks = pd.DataFrame(columns = ['First', 'Upper_second', 'Lower_second', 'Third', 'Fail', 'Course_code']) #temporary dataframe to be filled in with grade distribution

Fill in the temporary dataframe with grade distribution info:

In [5]:
for course in st_courses_list:
    url_course = f'https://www.lse.ac.uk/resources/calendar2020-2021/courseGuides/ST/2020_{course}.htm'
    r_course = requests.get(url_course)
    soup_course = BeautifulSoup(r_course.content,'lxml')
    
    if soup_course.find('div', {'id': 'exams-Content'}) == None:
        pass #skip courses which have no grade distribution info
    else:
        course_marks = pd.read_html(r_course.content)
        course_marks = course_marks[0].iloc[:, [1]].T #extract grade distribution info
        course_marks['Course_code'] = course #add a column with course code
        course_marks.rename(columns = {0: 'First', 1: 'Upper_second', 2: 'Lower_second', 3: 'Third', 4: 'Fail'}, inplace = True) #rename columns
        st_courses_marks = pd.concat([st_courses_marks, course_marks], ignore_index = True, axis = 0) #add grade distribution info to the temporary dataframe
        
st_courses_marks.set_index('Course_code', inplace = True)

Merge the dataframe from Step 3 with the temporary dataframe:

In [6]:
st_courses = st_courses.join(st_courses_marks)

**Step 5: Create a temporary dataframe with some key course characteristics and merge it with the dataframe from Step 4. The course characteristics are: total students in 2019/20, average class size in 2019/20, whether a course was capped in 2019/20 or not and course value**

Create a temporary dataframe:

In [7]:
st_courses_key_facts = pd.DataFrame(columns = ['Total_students_2019/20', 'Average_class_size_2019/20', 'Capped_2019/20', 'Value', 'Course_code'])

Define a function which ensures that there is no space before and after a `str`. This function will be used in the next code chunk. 

In [8]:
def remove_space(list_of_list):
    '''
    The function ensures that there is no extra space before and after a str, which is part of a list of list

    Parameters
    ----------
    list : list of list of str
        
    Returns
    -------
    list : list of list of str of the form: [['str', 'str', ..., 'str'], ..., [...]]
       The str which are part of list of list have no spaces before and after 
    '''    
    empty_list = []
    for sublist in list_of_list:
        empty_sublist = []
        for element in sublist:
            element = element.strip() #remove empty spaces before/after a str
            empty_sublist.append(element)
        empty_list.append(empty_sublist)
    return empty_list

Fill in the temporary dataframe with the info on the courses' key characteristics:

In [9]:
for course in st_courses_list:
    url_key_facts = f'https://www.lse.ac.uk/resources/calendar2020-2021/courseGuides/ST/2020_{course}.htm'
    r_key_facts = requests.get(url_key_facts)
    soup_key_facts = BeautifulSoup(r_key_facts.content,'lxml') 
    
    key_facts = soup_key_facts.find('div', {'id': 'keyFacts-Content'}).find_all_next('p')
    key_facts_list = [] #list to be populated with the key characteristics of a course
    
    for info in key_facts:
        key_facts_list.append(info.text.split(':')) #key characteristics of a course
    key_facts_list = key_facts_list[1 : -1] #delete the extra info on `Department` and `Personal development skills`   
    key_facts_list = remove_space(key_facts_list)
    key_facts_df = pd.DataFrame(key_facts_list).T #store the key characteristics in a dataframe
    key_facts_df = key_facts_df.iloc[[1], :] #remove the first row with the names of the key characteristics
    key_facts_df['Course_code'] = course #add a column with course code
    key_facts_df.rename(columns = {0: 'Total_students_2019/20', 1: 'Average_class_size_2019/20', 2: 'Capped_2019/20', 3: 'Value'}, inplace = True) #rename columns
    st_courses_key_facts = pd.concat([st_courses_key_facts, key_facts_df], ignore_index = True, axis = 0) #add key characteristics info to the temporary dataframe
     
st_courses_key_facts.set_index('Course_code', inplace = True)

Merge the dataframe from Step 4 with the temporary dataframe:

In [10]:
st_courses = st_courses.join(st_courses_key_facts)

**Step 6: Create a temporary dataframe with the course assessment information and merge it with the dataframe from Step 5. The course assessment types are: exam, coursework, project, assessment and presentation**

Create a temporary dataframe:

In [11]:
st_courses_assessment = pd.DataFrame(columns = ['exam', 'coursework', 'project', 'assessment', 'presentation', 'Course_code'])

Define a function which ensures that columns to be used for dataframe creation are all of the same length. This function will be used in the next code chunk. 

In [12]:
def populate_with_none(dictionary):
    '''
    The function ensures that the input dict has values of the same length. This ensures that the dictionary can be used for subsequent dataframe creation.  
    

    Parameters
    ----------
    dictionary : dict with str keys and list values. 
        The keys of the dict will be later used as column names and the values will be columns themselves. The values are either empty list or contain some str

    Returns
    -------
    dictionary : dict
       The values of the dict all have the same length

    '''    
    max_length = 0
    for i in dictionary.values():
        length = len(i)
        max_length = max(max_length, length) #dict value with the greatest length
    
    for keys, values in dictionary.items():
        while len(values) < max_length:
            dictionary[keys].append(None) #populate empty lists with `None`
    return dictionary

Fill in the temporary dataframe with the info on the courses' assessment:

In [13]:
for course in st_courses_list:
    url_assessment = f'https://www.lse.ac.uk/resources/calendar2020-2021/courseGuides/ST/2020_{course}.htm'
    r_assessment = requests.get(url_assessment)
    soup_assessment = BeautifulSoup(r_assessment.content,'lxml') 
    
    assessment_info = soup_assessment.find('div', {'id': 'assessment-Content'}).find('p').text.lower()
    assessment_data = {'exam': [], 
                      'coursework': [],
                      'project': [],
                      'assessment': [],
                      'presentation': [],
                      'Course_code': []} #dict to be populated with assessment percentages
    
    assessment_info_list = assessment_info.replace('(', ' ').replace(')', ' ').split() # turn `assessment_info` into a list and prepare for info extraction
    for i in assessment_info_list:
        assessment_aspect = [] #list to be populated with individual assessment info of a couse
        if '%' in i:
            percentage = i.strip(',%') #extract percentage of the final mark that a given assessment type accounts for
            idx = assessment_info_list.index(f'{i}') #index of the percentage
            assessment_type_idx = idx - 1 #index of the assessment type that corresponds to the percentage
            assessment_type = assessment_info_list[assessment_type_idx] #assessment type
            
            if assessment_type in assessment_data: #update the dict with assessment percentage
                assessment_data[assessment_type].append(percentage)
                   
    assessment_data['Course_code'].append(course) #add course code column
    assessment_data = populate_with_none(assessment_data) #ensure all columns are of the same length
    assessment_df = pd.DataFrame(assessment_data) #store assessment data in a temporary dataframe   
    st_courses_assessment = pd.concat([st_courses_assessment, assessment_df], ignore_index = True, axis = 0) #add key characteristics info to the temporary dataframe
     
st_courses_assessment.set_index('Course_code', inplace = True)

**Please note the following is assumed for the code chunk above: in the raw text from course web pages, 'assessment type' is always the word preceeding the percentage of the final mark that it accounts for.**

In [14]:
print(f'There are {st_courses_assessment.shape[0]} rows in `st_courses` and only {st_courses.shape[0]} rows in `st_courses`.')

There are 32 rows in `st_courses` and only 30 rows in `st_courses`.


Therefore, the data is checked to see why there are two extra rows:

In [15]:
st_courses_assessment.head(21)

,exam,coursework,project,assessment,presentation
Course_code,,,,,
ST101,None,40,60,None,None
ST102,75,None,None,None,None
NaN,25,None,None,None,None
ST102GC,75,None,None,25,None
ST107,100,None,None,None,None
ST108,80,None,None,20,None
ST115,None,40,60,None,None
ST201,80,20,None,None,None
ST202,100,None,None,None,None


There are two rows with `Course_code` = `None`. Having examined the information provided by the ST102 and ST303 web pages, I realized that the following rows should be combined:

* Row with `Course_code` = `ST102` and the subsequent row with `Course_code` = `None`
* Row with `Course_code` = `ST303` and the subsequent row with `Course_code` = `None`

The combination is justified as ST102 is 100% exam based, while ST303 is 100% project based. Given the scope of this study, I am not interested in which terms the assessments took place. 

In [16]:
st_courses_assessment.iloc[[1], [0]] = '100%' #ST102 is 100% exam-based
st_courses_assessment.iloc[[19], [2]] = '100%' #ST303 is 100% project-based
st_courses_assessment.drop([st_courses_assessment.index[2], st_courses_assessment.index[20]], inplace = True) #remove unnecessary rows

Merge the dataframe from Step 5 with the temporary dataframe:

In [17]:
st_courses = st_courses.join(st_courses_assessment)

**Step 7: Create a temporary dataframe with the course teaching information and merge it with the dataframe from Step 6**

Create a temporary dataframe with the info on the courses' teaching hours:

In [18]:
teaching_hours = [] #list to be populated with teaching hours

for course in st_courses_list:
    url_teaching = f'https://www.lse.ac.uk/resources/calendar2020-2021/courseGuides/ST/2020_{course}.htm'
    r_teaching = requests.get(url_teaching)
    soup_teaching = BeautifulSoup(r_teaching.content,'lxml') 
    
    teaching_info = soup_teaching.find('div', {'id': 'teaching-Content'}).find_all('p')[1].text
    teaching_info_list = teaching_info.split()
    
    if 'hours' in teaching_info:
        hours = 0 #to be updated according to a course's teaching hours amount
        for i in teaching_info_list:
            if i == 'hours':
                idx = teaching_info_list.index(i) #index of the word 'hours'
                hours_idx = idx - 1 #index of the hours value
                hours += int(teaching_info_list[hours_idx]) #update the number of teaching hours
                teaching_info_list = teaching_info_list[idx+1:] #remove all list elements until after the word 'hours'
        teaching_hours.append(hours) #store hours info           
    else:
        teaching_hours.append(None)
    
st_courses_teaching = pd.DataFrame({'Course_code': st_courses_list, 'Teaching_hours': teaching_hours})
st_courses_teaching.set_index('Course_code', inplace = True)

Merge the dataframe from Step 6 with the temporary dataframe:

In [19]:
st_courses = st_courses.join(st_courses_teaching)

**Step 8: Add two columns: 'Academic_year' and 'Level_of_study'**

Academic_year:

In [20]:
st_courses = st_courses.assign(Academic_year = ['2020/21' for i in range(30)])

Level_of_study:

In [21]:
level_of_study = ['Year 1' for i in range(6)] + ['Year 2' for i in range(9)] + ['Year 3' for i in range(15)]
st_courses = st_courses.assign(Level_of_study = level_of_study)

## Part 2: 2017/18 academic year

Repeat the 9 steps layed out in **Part 1** for the 2017/18 academic year. Some steps are changed slightly but the overall logic is preserved. Please note that fewer comments are made to avoid repetition.

**Step 1**

In [22]:
url_courses_list = 'https://www.lse.ac.uk/resources/calendar2017-2018/courseGuides/undergraduate.htm#generated-subheading20'
r_courses_list = requests.get(url_courses_list)
soup_courses_list = BeautifulSoup(r_courses_list.content,'lxml')
list_of_courses = pd.read_html(r_courses_list.content) #list of all LSE courses

**Step 2**

In [23]:
st_courses_2017 = list_of_courses[19] #extract ST courses
st_courses_2017['Course_code'] = st_courses_2017['ST'].str.split().str[:1].str.join('') #add a column with course code info
st_courses_2017['Course_name'] = st_courses_2017['ST'].str.split().str[1:].str.join(' ') #add a column with course name info
st_courses_2017.drop('ST', axis = 1, inplace = True) #remove the 'ST' column to avoid repeating info
st_courses_2017.set_index('Course_code', inplace = True) #set course code as index

**Step 3**

In [24]:
#Get a list of all ST courses and create a temporary dataframe:
st_courses_list = st_courses_2017.index.tolist() 
st_courses_marks = pd.DataFrame(columns = ['First', 'Upper_second', 'Lower_second', 'Third', 'Fail', 'Course_code']) #temporary dataframe to be filled in with grade distribution

**Step 4**

In [25]:
#Fill in the temporary dataframe with grade distribution info:
for course in st_courses_list:
    url_course = f'https://www.lse.ac.uk/resources/calendar2017-2018/courseGuides/ST/2017_{course}.htm'
    r_course = requests.get(url_course)
    soup_course = BeautifulSoup(r_course.content,'lxml')
    
    if soup_course.find('div', {'id': 'exams-Content'}) == None:
        pass #skip courses which have no grade distribution info
    else:
        course_marks = pd.read_html(r_course.content)
        course_marks = course_marks[0].iloc[:, [1]].T #extract grade distribution info
        course_marks['Course_code'] = course #add a column with course code
        course_marks.rename(columns = {0: 'First', 1: 'Upper_second', 2: 'Lower_second', 3: 'Third', 4: 'Fail'}, inplace = True) #rename columns
        st_courses_marks = pd.concat([st_courses_marks, course_marks], ignore_index = True, axis = 0) #add grade distribution info to the temporary dataframe
        
st_courses_marks.set_index('Course_code', inplace = True)

In [26]:
#Merge the dataframe from Step 3 with the temporary dataframe:
st_courses_2017 = st_courses_2017.join(st_courses_marks)

**Step 5**

In [27]:
#Create a temporary dataframe:
st_courses_key_facts = pd.DataFrame(columns = ['Total_students_2016/17', 'Average_class_size_2016/17', 'Capped_2016/17', 'Lecture_capture', 'Value', 'Course_code'])

In [28]:
#Fill in the temporary dataframe with the info on the courses' key characteristics:
for course in st_courses_list:
    url_key_facts = f'https://www.lse.ac.uk/resources/calendar2017-2018/courseGuides/ST/2017_{course}.htm'
    r_key_facts = requests.get(url_key_facts)
    soup_key_facts = BeautifulSoup(r_key_facts.content,'lxml') 
    
    key_facts = soup_key_facts.find('div', {'id': 'keyFacts-Content'}).find_all_next('p')
    key_facts_list = [] #list to be populated with the key characteristics of a course
    
    for info in key_facts:
        key_facts_list.append(info.text.split(':')) #key characteristics of a course
        
    key_facts_list = key_facts_list[1 : -1] #delete the extra info on `Department` and `Personal development skills`   
    key_facts_list = remove_space(key_facts_list)
    key_facts_df = pd.DataFrame(key_facts_list).T #store the key characteristics in a dataframe
    key_facts_df = key_facts_df.iloc[[1], :] #remove the first row with the names of the key characteristics
    key_facts_df['Course_code'] = course #add a column with course code
    key_facts_df.rename(columns = {0: 'Total_students_2016/17', 1: 'Average_class_size_2016/17', 2: 'Capped_2016/17', 3: 'Lecture_capture', 4: 'Value'}, inplace = True) #rename columns
    st_courses_key_facts = pd.concat([st_courses_key_facts, key_facts_df], ignore_index = True, axis = 0) #add key characteristics info to the temporary dataframe
    

st_courses_key_facts.set_index('Course_code', inplace = True)

In [29]:
print(f'There are {st_courses_key_facts.shape[1]} columns in the temporary data frame, which is alarming')

There are 33 columns in the temporary data frame, which is alarming


Therefore, we require an overview of the data:

In [30]:
st_courses_key_facts.head(3)

,Total_students_2016/17,Average_class_size_2016/17,Capped_2016/17,Lecture_capture,Value,5,6,7,8,9,...,23,24,25,26,27,28,29,30,31,32
Course_code,,,,,,,,,,,,,,,,,,,,,
ST102,588,15,No,Yes (MT & LT),One Unit,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ST107,364,16,No,Yes (LT),Half Unit,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ST108,7,11,No,One Unit,None,None,None,None,65%,None,...,None,None,None,None,None,None,None,None,None,NaN


Having examined the web page for `ST108`, I found out that there is extra information provided for this course which is not helpful for the study. Hence, I choose to remove columns `5` to `32`:

In [31]:
for i in range(5, 33):
    st_courses_key_facts.drop(i, axis = 1, inplace = True) #remove the 'i' column which contains unnecessary info

In [32]:
#Merge the dataframe from Step 4 with the temporary dataframe:
st_courses_2017 = st_courses_2017.join(st_courses_key_facts)

**Step 6**

In [33]:
#Create a temporary dataframe:
st_courses_assessment = pd.DataFrame(columns = ['exam', 'coursework', 'project', 'assessment', 'presentation', 'Course_code'])

In [34]:
#Fill in the temporary dataframe with the info on the courses' assessment:
for course in st_courses_list:
    url_assessment = f'https://www.lse.ac.uk/resources/calendar2017-2018/courseGuides/ST/2017_{course}.htm'
    r_assessment = requests.get(url_assessment)
    soup_assessment = BeautifulSoup(r_assessment.content,'lxml') 
    
    assessment_info = soup_assessment.find('div', {'id': 'assessment-Content'}).find('p').text.lower()
    assessment_data = {'exam': [], 
                      'coursework': [],
                      'project': [],
                      'assessment': [],
                      'presentation': [],
                      'Course_code': []} #dict to be populated with assessment percentages
    
    assessment_info_list = assessment_info.replace('(', ' ').replace(')', ' ').split() # turn `assessment_info` into a list and prepare for info extraction
    for i in assessment_info_list:
        assessment_aspect = [] #list to be populated with individual assessment info of a couse
        if '%' in i:
            percentage = i.strip(',%') #extract percentage of the final mark that a given assessment type accounts for
            idx = assessment_info_list.index(f'{i}') #index of the percentage
            assessment_type_idx = idx - 1 #index of the assessment type that corresponds to the percentage
            assessment_type = assessment_info_list[assessment_type_idx] #assessment type
            
            if assessment_type in assessment_data: #update the dict with assessment percentage
                assessment_data[assessment_type].append(percentage)
                   
    assessment_data['Course_code'].append(course) #add course code column
    assessment_data = populate_with_none(assessment_data) #ensure all columns are of the same length
    assessment_df = pd.DataFrame(assessment_data) #store assessment data in a temporary dataframe   
    st_courses_assessment = pd.concat([st_courses_assessment, assessment_df], ignore_index = True, axis = 0) #add key characteristics info to the temporary dataframe
     
st_courses_assessment.set_index('Course_code', inplace = True)

In [35]:
print(f'There are {st_courses_assessment.shape[0]} rows in `st_courses` and only {st_courses_2017.shape[0]} rows in `st_courses`.')

There are 25 rows in `st_courses` and only 23 rows in `st_courses`.


Therefore, the data is checked to see why there are two extra rows:

In [36]:
st_courses_assessment.head(17)

,exam,coursework,project,assessment,presentation
Course_code,,,,,
ST102,75,None,None,None,None
NaN,25,None,None,None,None
ST107,100,None,None,None,None
ST108,80,None,None,20,None
ST201,80,20,None,None,None
ST202,100,None,None,None,None
ST205,80,20,None,None,None
ST206,100,None,None,None,None
ST211,50,None,50,None,None


There are two rows with `Course_code` = `None`. Similarly to **Part 1, Step 6**, I combine the following rows:

* Row with `Course_code` = `ST102` and the subsequent row with `Course_code` = `None`
* Row with `Course_code` = `ST303` and the subsequent row with `Course_code` = `None`

I further note that `ST303` should be 100% project-based according to its course page. 

In [37]:
st_courses_assessment.iloc[[0], [0]] = '100%' #ST102 is 100% exam-based
st_courses_assessment.iloc[[15], [2]] = '100%' #ST303 is 100% project-based
st_courses_assessment.drop([st_courses_assessment.index[1], st_courses_assessment.index[16]], inplace = True) #remove unnecessary rows

In [38]:
#Merge the dataframe from Step 5 with the temporary dataframe:
st_courses_2017 = st_courses_2017.join(st_courses_assessment)

**Step 7**

In [39]:
#Create a temporary dataframe with the info on the courses' teaching hours:
teaching_hours = [] #list to be populated with teaching hours

for course in st_courses_list:
    url_teaching = f'https://www.lse.ac.uk/resources/calendar2017-2018/courseGuides/ST/2017_{course}.htm'
    r_teaching = requests.get(url_teaching)
    soup_teaching = BeautifulSoup(r_teaching.content,'lxml') 
    
    teaching_info = soup_teaching.find('div', {'id': 'teaching-Content'}).find('p').text
    teaching_info_list = teaching_info.split()
    
    if 'hours' in teaching_info:
        hours = 0 #to be updated according to a course's teaching hours amount
        for i in teaching_info_list:
            if i == 'hours':
                idx = teaching_info_list.index(i) #index of the word 'hours'
                hours_idx = idx - 1 #index of the hours value
                hours += int(teaching_info_list[hours_idx]) #update the number of teaching hours
                teaching_info_list = teaching_info_list[idx+1:] #remove all list elements until after the word 'hours'
        teaching_hours.append(hours) #store hours info           
    else:
        teaching_hours.append(None)
    
st_courses_teaching = pd.DataFrame({'Course_code': st_courses_list, 'Teaching_hours': teaching_hours})
st_courses_teaching.set_index('Course_code', inplace = True)

In [40]:
#Merge the dataframe from Step 6 with the temporary dataframe:
st_courses_2017 = st_courses_2017.join(st_courses_teaching)

**Step 8**

In [41]:
#Academic_year:
st_courses_2017 = st_courses_2017.assign(Academic_year = ['2017/18' for i in range(23)])
#Level_of_study:
level_of_study = ['Year 1' for i in range(3)] + ['Year 2' for i in range(8)] + ['Year 3' for i in range(12)]
st_courses_2017 = st_courses_2017.assign(Level_of_study = level_of_study)

# Part 3: store the dataframes obtained in Parts 1 and 2:

From **Part 1**:

In [42]:
st_courses.to_csv('data/st_courses.csv')

From **Part 2**:

In [43]:
st_courses_2017.to_csv('data/st_courses_2017.csv')